# 02. 베이스라인

**목적: 기준점을 만든다.**

나중에 "정확도 0.98" 을 얻었을 때, 그게 모델이 잘한 건지 문제가 원래 쉬운 건지
구분할 자가 필요하다. 그 자가 베이스라인이다.

| 기준선 | 뜻 |
|---|---|
| 0.25 | 무작위로 찍기 (클래스 4개, 균형) |
| **A. 밑바닥 CNN** | 사전학습 없이 우리가 만든 CNN — *데이터만으로 얼마나 되나* |
| **B. 백본 고정** | ImageNet 특징 + 선형분류기만 학습 — *ImageNet 지식만으로 얼마나 되나* |
| **C. 전체 미세조정** | 백본까지 전부 학습 — *본 게임* |

A→C 의 차이가 **전이학습의 순수 이득**이고, B→C 의 차이가 **"ImageNet 특징만으로는 부족하다"의 크기**다.

이 노트북에서는 학습 루프를 **직접 펼쳐서** 한 번 보여준 뒤,
이후 실험은 같은 내용을 담은 `wbc.train_model()` 로 돌린다.

In [ ]:
import os, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn

import wbc
wbc.use_korean_font()

IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS     = 20
wbc.NUM_WORKERS = 4      # 윈도우 주피터에서 DataLoader 오류가 나면 0 으로

print('장치:', wbc.device)
if wbc.device.type == 'cuda':
    print('GPU :', torch.cuda.get_device_name(0))
else:
    print('⚠ CPU 로 도는 중이다. GPU 가 안 잡히면 3분 예산을 맞출 수 없다.')

## 2-1. 데이터로더

TRAIN 폴더를 **층화분할**해 train / val 로 나눈다(클래스 비율 유지).
**TEST 폴더는 06 노트북 전까지 열지 않는다.**

같은 폴더를 두 번 여는 요령(증강 있는 것 / 없는 것)은 수업 5장 그대로이고,
거기서 지적됐던 "검증셋에 증강이 걸려 있는 버그"는 고쳐 놓았다.

In [ ]:
train_loader, val_loader, test_loader = wbc.make_loaders(
    preset='flip', image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, val_ratio=0.2, seed=42)

print(f'train {len(train_loader.dataset)}장 / val {len(val_loader.dataset)}장 / test {len(test_loader.dataset)}장')
xb, yb = next(iter(train_loader))
print('배치 모양', tuple(xb.shape), '라벨 예시', yb[:8].tolist())
print('클래스 순서', wbc.CLASS_NAMES)

## 2-2. 베이스라인 A — 밑바닥 CNN

수업 3장에서 만든 구조(Conv-BN-ReLU ×2 + MaxPool 을 4단)를 4클래스로 바꾼 것이다.
**`GAP → Linear` 로 끝나는 구조**라는 점이 중요하다. 07 노트북에서 CAM 을 근사 없이 뽑을 수 있다.

(아래는 `wbc.py` 의 `SimpleCNN` 과 같은 구조를 노트북에 그대로 펼쳐 쓴 것이다.)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4, width=32):
        super().__init__()
        def block(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.MaxPool2d(2))
        self.features = nn.Sequential(
            block(3, width), block(width, width*2), block(width*2, width*4), block(width*4, width*8))
        self.pool = nn.AdaptiveAvgPool2d(1)         # 전역 평균 풀링
        self.head = nn.Linear(width*8, num_classes) # 선형 한 층 -> CAM 가능

    def forward_features(self, x):
        return self.features(x)                     # (B, C, h, w)

    def forward(self, x):
        return self.head(self.pool(self.forward_features(x)).flatten(1))

m = SimpleCNN()
with torch.no_grad():
    out = m(torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE))
print('출력 모양', tuple(out.shape))
print(f'파라미터 {sum(p.numel() for p in m.parameters()):,}')

### 학습 루프를 직접 펼쳐 보기

수업 4·5장의 `train_model` 과 같은 내용이다. 이진분류에서 바뀐 곳은 두 군데다.

| | 4·5장 (이진) | 여기 (다중) |
|---|---|---|
| 손실 | `BCEWithLogitsLoss` | **`CrossEntropyLoss`** |
| 출력 | `model(x).squeeze(1)` → 로짓 1개 | **`model(x)` → 로짓 4개, squeeze 안 함** |
| 예측 | `sigmoid(logit) > 0.5` | **`logits.argmax(1)`** |
| 라벨 | `.float()` 필요 | **`long` 그대로** (CrossEntropy 가 정수 라벨을 받는다) |

시간을 매 에폭 재서 **3분 예산**을 넘는지 바로 확인한다.

In [ ]:
model = SimpleCNN().to(wbc.device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(1, 3):                      # 구조 확인용으로 2 에폭만 펼쳐 본다
    model.train()
    run_loss, run_correct, n = 0.0, 0, 0
    t0 = time.time()

    for xb, yb in train_loader:
        xb, yb = xb.to(wbc.device), yb.to(wbc.device)
        optimizer.zero_grad()
        logits = model(xb)                     # (B, 4)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        run_loss += loss.item() * len(yb)
        run_correct += (logits.argmax(1) == yb).sum().item()
        n += len(yb)

    m_val = wbc.evaluate(model, val_loader, criterion)
    sec = time.time() - t0
    print(f'epoch {epoch} | train loss {run_loss/n:.4f} acc {run_correct/n:.4f} | '
          f'val loss {m_val["loss"]:.4f} acc {m_val["accuracy"]:.4f} macroF1 {m_val["macro_f1"]:.4f} | '
          f'{sec:.0f}s' + ('  ⚠3분초과' if sec > 180 else ''))

이제 같은 루프에 **조기종료 · 체크포인트 저장 · AMP · 실험기록**을 붙인
`wbc.run_experiment()` 로 본 학습을 돌린다. 결과는 `results/runs.csv` 에 자동 저장된다.

In [ ]:
res_a = wbc.run_experiment('A_밑바닥CNN', model_name='simplecnn', preset='flip',
                           image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, lr=1e-3,
                           epochs=EPOCHS, pretrained=False, patience=5)
if res_a: wbc.plot_history(res_a['history'], 'A. 밑바닥 CNN'); plt.show()

## 2-3. 베이스라인 B — 백본 고정 (수업 4장 방식)

ImageNet 으로 학습된 ResNet-18 을 가져오되 **백본은 얼리고 마지막 선형층만** 학습한다.
학습 파라미터가 2,052개(512×4+4)뿐이라 매우 빠르다.

질문: **"ImageNet 이 배운 일반 사진의 특징만으로 백혈구가 구분되는가?"**

In [ ]:
res_b = wbc.run_experiment('B_백본고정', model_name='resnet18', preset='flip',
                           image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, lr=1e-3,
                           epochs=EPOCHS, pretrained=True, freeze=True, patience=5)

## 2-4. 베이스라인 C — 전체 미세조정 (수업 5장 방식)

같은 ResNet-18 을 **전부 학습**한다. 5장에서 배운 대로 **학습률을 작게(3e-4)** 잡는다.
사전학습 가중치를 크게 흔들면 이미 배운 것을 잃기 때문이다.

현미경 염색 이미지는 ImageNet 과 결이 다르므로, 5장의 X-ray 와 같은 이유로
**미세조정이 유리할 것**으로 예상한다.

In [ ]:
res_c = wbc.run_experiment('C_미세조정', model_name='resnet18', preset='flip',
                           image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, lr=3e-4,
                           epochs=EPOCHS, pretrained=True, freeze=False, patience=5)
if res_c: wbc.plot_history(res_c['history'], 'C. ResNet-18 미세조정'); plt.show()

## 2-5. 세 베이스라인 비교

In [ ]:
t = wbc.runs_table()
base = t[t.run_id.isin(['A_밑바닥CNN', 'B_백본고정', 'C_미세조정'])]
display(base[['run_id', 'params_M', 'best_epoch', 'epoch_sec',
              'val_accuracy', 'val_macro_f1', 'val_auc']].round(4))
print('무작위 기준선 = 0.2500')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(base.run_id, base.val_macro_f1, color=['tab:gray', 'tab:orange', 'tab:blue'])
ax[0].axhline(0.25, ls='--', c='r'); ax[0].text(-.4, .26, '무작위 0.25', fontsize=9, color='r')
ax[0].set_ylabel('검증 macro-F1'); ax[0].set_title('성능'); ax[0].tick_params(axis='x', rotation=15)
ax[1].bar(base.run_id, base.epoch_sec, color='tab:green')
ax[1].axhline(180, ls='--', c='r'); ax[1].set_ylabel('에폭당 시간(초)')
ax[1].set_title('비용 (빨간선 = 3분)'); ax[1].tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

### 여기서 읽어야 할 것

1. **A vs C** — 전이학습의 순수 이득.
2. **B vs C** — B 가 C 에 크게 못 미치면 "현미경 이미지는 ImageNet 특징으로 표현되지 않는다"는 뜻이고,
   이는 수업 5장의 **"도메인이 다르면 미세조정"** 원칙과 일치한다.
3. **에폭당 시간** — B 는 역전파가 헤드에만 흘러 훨씬 빠르다. 같은 예산이면 더 많이 돌릴 수 있다는 뜻이기도 하다.

> A 와 C 의 차이가 통계적으로 유의한지는 **08 노트북의 가설검정 1(McNemar)** 에서 확인한다.
> 지금은 "숫자가 다르다"까지만 말한다.

## 2-6. 에폭은 몇으로 잡을 것인가

"충분한 에폭"은 감이 아니라 **검증 곡선**으로 정한다.

- 검증 손실이 더 내려가지 않고 오르기 시작하는 지점 = 과적합 시작점
- `train_model` 은 검증 macro-F1 이 가장 좋은 시점의 가중치를 저장하고(`저장` 표시),
  `patience` 에폭 동안 개선이 없으면 **조기종료**한다
- 따라서 `epochs` 는 "최대 몇 번까지 허용"이고, 실제로 쓰인 값은 **`best_epoch`** 다

In [ ]:
plt.figure(figsize=(7, 4))
for name, r in [('A 밑바닥', res_a), ('B 고정', res_b), ('C 미세조정', res_c)]:
    if r is None: continue
    plt.plot(range(1, len(r['history']['val_loss'])+1), r['history']['val_loss'], marker='o', label=name)
plt.xlabel('epoch'); plt.ylabel('검증 손실'); plt.title('에폭 결정 — 검증 손실이 꺾이는 지점')
plt.legend(); plt.grid(alpha=.3); plt.show()

for name, r in [('A_밑바닥CNN', res_a), ('C_미세조정', res_c)]:
    if r is None: continue
    h = r['history']
    msg = '← 최대치에 붙었다. 에폭을 늘려야 한다' if h['best_epoch'] >= EPOCHS - 1 else '← 이 지점에서 수렴'
    print(f"{name:14s} best_epoch = {h['best_epoch']:2d} / 최대 {EPOCHS}   에폭당 {h['epoch_sec_mean']:.0f}s  {msg}")

## 02 정리

- 기준선 셋(0.25 / 밑바닥 / 고정)과 본 게임(미세조정)의 숫자가 나왔다
- 학습 루프의 내용과, 이진분류(4·5장)에서 다중분류로 바뀐 지점을 확인했다
- 에폭 수는 조기종료가 잡아주며, `best_epoch` 로 확인한다

→ 다음: **03_이미지전처리.ipynb** — 지금은 `flip` 만 썼다. 전처리를 바꾸면 얼마나 달라지는가?